# Income & expenses forecasting: analysis notebook

This notebook walks through the pipeline in `src/` on the real data: EDA → features → clusters →
training and hold-out evaluation → August forecasts. It calls the package code directly, so it shows
exactly what the CLI and API do.

**Running it efficiently**
* `RETRAIN = False` (default) loads `models/model_bundle.joblib` and `results/holdout_predictions.csv`
  from `uv run income train`. The whole notebook then runs in about 30 s.
* `RETRAIN = True` re-runs the full protocol: about 30 s with `N_TRIALS = 0` (default
  hyper-parameters), about 4 min with `N_TRIALS = 25` (what produced the shipped model).
* Launch with `uv run jupyter lab notebooks/analysis.ipynb` (after `uv sync --all-extras`).

In [1]:
RETRAIN = False   # True: re-run hold-out evaluation + final fit
N_TRIALS = 25     # Optuna trials per model when RETRAIN (0 = fast defaults)

In [2]:
import json, logging
import pandas as pd
from IPython.display import Markdown, display

from src import plotter
from src.bundle import load_bundle, save_bundle
from src.cluster import cluster_users
from src.config import get_paths
from src.eda import perform_eda
from src.evaluate import bootstrap_mae_diff, metrics_table, per_cluster_table
from src.predict import predict_next_month
from src.preprocessing import build_feature_frame, load_lookups, read_transactions
from src.train_model import TrainConfig, run_training

logging.basicConfig(level=logging.WARNING)
pd.set_option("display.float_format", "{:,.1f}".format)
paths = get_paths()
outdir_ = str(paths.img_dir) + "/"

lookups = load_lookups()
tx = read_transactions(paths.training_csv)
print(f"{len(tx):,} transactions, {tx.user_id.nunique():,} users, "
      f"{tx.transaction_date.min().date()} → {tx.transaction_date.max().date()}")

408,546 transactions, 10,000 users, 2016-02-01 → 2016-07-31


## 1. Exploratory data analysis

**What to expect:** amounts are bounded (3–1,166 €) and right-skewed. There are more outgoing
than incoming transactions. About 37% of transactions have no MCC (transfers), so `Unknown` is the
largest category. Active users grow each month (accounts opening), which is why the panel starts each
user at their *first* month rather than at February.

In [3]:
summary = perform_eda(tx, lookups, outdir_ + "eda/")
print(json.dumps({k: summary[k] for k in ("flow_share", "unknown_category_share", "ks_in_vs_out")}, indent=1))
display(Markdown("![](../img/eda/eda_overview.png)"))

{
 "flow_share": {
  "Out": 0.823,
  "In": 0.177
 },
 "unknown_category_share": 0.368,
 "ks_in_vs_out": {
  "stat": 0.41221849574806246,
  "p": 0.0
 }
}


![](../img/eda/eda_overview.png)

## 2. Load or train the model

`run_training` is leakage-free. It (i) splits **users** 80/20; (ii) fits clusters and models on the
training users using months **before July**; (iii) forecasts **July** for the unseen hold-out users from
their Feb–Jun history; (iv) refits on everyone and every month, so production forecasts **August**.

In [4]:
if RETRAIN:
    bundle, holdout = run_training(tx, lookups, TrainConfig(n_trials=N_TRIALS))
    save_bundle(bundle, paths.model_path)
    paths.results_dir.mkdir(exist_ok=True)
    holdout.to_csv(paths.results_dir / "holdout_predictions.csv", index=False)
else:
    holdout_csv = paths.results_dir / "holdout_predictions.csv"
    if not holdout_csv.exists():
        raise FileNotFoundError(f"{holdout_csv} missing: run `uv run income train` or set RETRAIN = True")
    bundle = load_bundle(paths.model_path)
    holdout = pd.read_csv(holdout_csv)

feats = cluster_users(build_feature_frame(tx, lookups), bundle.clusterer)
print("trained through", bundle.trained_through, "| features:", len(bundle.feature_columns))

trained through 2016-07 | features: 31


## 3. Toy example: one real user per cluster

Concept: each row is a **user-month**. Features at month *t* use only months ≤ *t* (lags, 3-month rolling
means, per-category spending). The target is the *next* month's total. Below, the user nearest the median
monthly expenses of each cluster.

**What to expect:** `income_lag1` in a row equals the previous row's `income`. `income_roll3_mean`
is the mean of up to three visible months. Clusters separate users mainly by activity level and by
whether a salary-like inflow is present.

In [5]:
prof = feats.groupby("user_id").agg(cluster=("cluster", "first"), med_exp=("expenses", "median"))
toy_users = (prof.assign(d=lambda d: (d.med_exp - d.groupby("cluster").med_exp.transform("median")).abs())
                 .sort_values("d").groupby("cluster").head(1).sort_values("cluster").index.tolist())
cols = ["user_id", "cluster", "period", "income", "expenses", "tx_count", "income_lag1", "income_roll3_mean",
        "expenses_roll3_mean"]
display(feats.loc[feats.user_id.isin(toy_users), cols].sort_values(["cluster", "period"]))

,user_id,cluster,period,income,expenses,tx_count,income_lag1,income_roll3_mean,expenses_roll3_mean
15523,47f0ae63f73586728c85687e0ce7226b,0,2016-02,0.0,18.0,2,0.0,0.0,18.0
15524,47f0ae63f73586728c85687e0ce7226b,0,2016-03,0.0,0.0,0,0.0,0.0,9.0
15525,47f0ae63f73586728c85687e0ce7226b,0,2016-04,76.0,17.0,3,0.0,25.3,11.7
15526,47f0ae63f73586728c85687e0ce7226b,0,2016-05,0.0,123.0,5,76.0,25.3,46.7
15527,47f0ae63f73586728c85687e0ce7226b,0,2016-06,116.0,316.0,14,0.0,64.0,152.0
15528,47f0ae63f73586728c85687e0ce7226b,0,2016-07,194.0,113.0,13,116.0,103.3,184.0
3965,129b1ff718dc2e7bb3067063e81716f4,1,2016-02,530.0,133.0,17,0.0,530.0,133.0
3966,129b1ff718dc2e7bb3067063e81716f4,1,2016-03,218.0,405.0,23,530.0,374.0,269.0
3967,129b1ff718dc2e7bb3067063e81716f4,1,2016-04,816.0,277.0,14,218.0,521.3,271.7
3968,129b1ff718dc2e7bb3067063e81716f4,1,2016-05,190.0,624.0,29,816.0,408.0,435.3


In [6]:
plotter.plot_clusters(feats, outdir_)
display(Markdown("![](../img/clusters.png)"))

![](../img/clusters.png)

## 4. Hold-out evaluation (July, unseen users)

The model is compared with two naive forecasts: **last month** (persistence) and the **3-month mean**.

**What to expect, and how to tell if something is wrong:**
* Income is lumpy (salary or nothing; many users have no inflow in a given month), so R² around 0.3–0.4 is
  realistic. Expenses are smoother, so R² is around 0.5.
* The model should clearly beat *last month* on every metric. Against the *3-month mean* it should win on
  RMSE and R², since a squared-error model estimates the conditional **mean**. On income MAE it can lose,
  because MAE rewards the conditional **median** and the median of a lumpy income is often 0.
* Warning signs: R² > 0.9 (leakage, e.g. July data in the features), the model losing to *last month*
  (broken features or targets), or residuals far from 0 (biased model).

In [7]:
m = metrics_table(holdout)
display(m.pivot(index="method", columns="target", values=["MAE", "RMSE", "R2"]).round(2))
display(pd.DataFrame([bootstrap_mae_diff(holdout, t) for t in ("income", "expenses")]).round(1))

MAE            RMSE              R2       
target     expenses income expenses income expenses income
method                                                    
last_month    151.3  159.9    337.5  314.9      0.3   -0.0
mean_3m       141.0  144.6    277.1  261.1      0.5    0.3
model         138.7  149.8    266.1  251.5      0.5    0.4

,target,baseline,mae_diff,ci95_low,ci95_high
0,income,mean_3m,5.2,0.1,10.1
1,expenses,mean_3m,-2.4,-8.0,3.3


In [8]:
plotter.plot_evaluation(holdout, outdir_)
display(Markdown("![](../img/evaluation_overview.png)"))
display(per_cluster_table(holdout).round(1))

![](../img/evaluation_overview.png)

,cluster,target,MAE,RMSE,R2,n,MAE_mean_3m
0,0,income,130.9,193.5,0.0,682,121.4
1,0,expenses,117.5,191.4,0.1,682,115.6
2,1,income,298.8,396.3,0.2,558,290.8
3,1,expenses,278.3,428.2,0.4,558,290.4
4,2,income,29.5,100.8,-0.1,196,29.5
5,2,expenses,63.3,170.1,0.3,196,62.2
6,3,income,43.1,99.2,-0.1,331,42.0
7,3,expenses,28.9,68.7,-0.0,331,28.6
8,4,income,90.1,167.7,0.3,190,95.9
9,4,expenses,73.3,145.6,0.1,190,71.0


### Toy example on the hold-out set: one unseen user per cluster

**What to expect:** forecasts track each user's recent level. Large misses come from one-off events
(a single large transfer) that no history-based model can anticipate.

In [9]:
hold_toy = holdout.groupby("cluster").apply(
    lambda g: g.iloc[(g.actual_expenses - g.actual_expenses.median()).abs().argsort()[:1]], include_groups=False)
display(hold_toy[["user_id", "income", "actual_income", "predicted_income", "mean_3m_income",
                  "expenses", "actual_expenses", "predicted_expenses", "mean_3m_expenses"]]
        .rename(columns={"income": "june_income", "expenses": "june_expenses"}))

,,user_id,june_income,actual_income,predicted_income,mean_3m_income,june_expenses,actual_expenses,predicted_expenses,mean_3m_expenses
cluster,,,,,,,,,,
0,1062,89bd78708c85ed0c0269e34c1bb97e77,0.0,0.0,113.9,267.0,71.0,60.0,109.6,170.0
1,1212,9d9c3b3f7cde26cf65bf311ccd26ee75,813.0,737.0,635.3,775.0,573.0,384.0,615.4,540.7
2,7,01274b958a92d6b7a62f75d164347b97,0.0,0.0,0.3,0.0,0.0,0.0,24.8,51.3
3,732,5ecacc135ccb7c4164b9d7eddda3c371,0.0,0.0,10.8,12.7,4.0,0.0,9.9,11.0
4,1840,f1adfba70c9744f63f4cd25c6cd6293b,383.0,264.0,253.4,211.3,12.0,13.0,197.3,5.7


## 5. What drives the forecasts

**What to expect:** recent levels dominate (`*_roll3_mean`, current-month `income`/`expenses`). Category
spending averages add little beyond the totals.

In [10]:
plotter.plot_feature_importance(bundle, outdir_)
plotter.plot_shap_summary(bundle, feats, outdir_)
display(Markdown("![](../img/feature_importance.png)"))
display(Markdown("![](../img/shap_summary.png)"))

/Users/abdulibrahim/Documents/gitHub/income/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


![](../img/feature_importance.png)

![](../img/shap_summary.png)

## 6. Production forecast: August 2016 for every user

Same call as `income predict` and `POST /predict`. Breakdowns split each total by the user's last 3 months
of category shares, so they sum to the total.

In [11]:
aug = predict_next_month(bundle, tx)
print(aug.forecast_month.iloc[0], "forecasts for", len(aug), "users")
display(aug[aug.user_id.isin(toy_users)][["user_id", "cluster", "predicted_income", "predicted_expenses",
                                          "predicted_expenses_breakdown"]])
display(aug[["predicted_income", "predicted_expenses"]].describe().T)

2016-08 forecasts for 10000 users


,user_id,cluster,predicted_income,predicted_expenses,predicted_expenses_breakdown
725,129b1ff718dc2e7bb3067063e81716f4,1,393.3,427.6,"{'ATM': 57.23, 'Automotive/Gas': 16.31, 'Busin..."
2868,47f0ae63f73586728c85687e0ce7226b,0,102.4,124.4,"{'ATM': 49.14, 'Automotive/Gas': 35.84, 'Busin..."
2920,499fd9637d7b8d3cf928f28d6a6ed41f,3,10.3,8.3,{'Unknown': 8.27}
3100,4e273ca5ae5fdf50b684c85886a545ee,4,68.6,50.2,{'ATM': 50.25}
4491,720c37775368b7aca827e7671a5d9ea8,2,0.2,13.9,{'Travel/Hotel': 13.89}


,count,mean,std,min,25%,50%,75%,max
predicted_income,"10,000.0",172.2,206.1,0.2,19.9,103.6,252.1,"2,519.5"
predicted_expenses,"10,000.0",207.6,306.3,5.1,18.4,96.1,277.0,"3,283.7"
